# Web Scraping Workshop 🕷️

## Huisartsen in Nederland: wie scoort het best?

In deze workshop leren we de basics van web scraping door huisarts-beoordelingen van **Zorgkaart Nederland** te verzamelen en analyseren.

### Wat gaan we doen?
1. Een webpagina ophalen met `requests`
2. HTML parsen met `BeautifulSoup`
3. Data extraheren uit de pagina
4. Meerdere pagina's en steden scrapen
5. De data analyseren en visualiseren met `pandas` en `matplotlib`

In [ ]:
%pip install requests beautifulsoup4 lxml pandas matplotlib

## Stap 1: Een webpagina ophalen

Web scraping begint altijd hetzelfde: je stuurt een HTTP-request naar een URL en krijgt HTML terug. Precies wat je browser ook doet — maar dan zonder de mooie plaatjes.

We gebruiken de library `requests` om de pagina van huisartsen in Amsterdam op te halen.

In [ ]:
import requests

url = "https://www.zorgkaartnederland.nl/huisarts/amsterdam"

# We doen ons voor als een browser, anders blokkeert de server ons soms
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")  # 200 = succes!
print(f"Grootte: {len(response.text):,} tekens")
print(f"\nEerste 500 tekens:\n{response.text[:500]}")

## Stap 2: HTML parsen met BeautifulSoup

We hebben nu ~100.000 tekens aan HTML. Daar handmatig doorheen zoeken is geen optie. **BeautifulSoup** parsed de HTML en laat ons zoeken met CSS selectors — net als in de browser DevTools.

> **💡 Tip:** Open https://www.zorgkaartnederland.nl/huisarts/amsterdam in je browser, klik rechtermuisknop op een naam → *Inspecteren*. Zo zie je de HTML-structuur.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "lxml")

# Elke huisarts staat in een <div class="filter-result">
cards = soup.select(".filter-result")
print(f"Aantal huisarts-kaarten op de pagina: {len(cards)}")

# Laten we de eerste kaart bekijken
print(f"\n--- HTML van de eerste kaart ---\n")
print(cards[0].prettify())

## Stap 3: Data uit één kaart halen

Elke kaart heeft deze structuur:
```
filter-result
├── filter-result__name          → naam (+ link naar profiel)
├── filter-result__role          → "Huisarts"
├── filter-result__places        → praktijknaam + stad (+ lat/lon!)
├── filter-result__score         → cijfer (1-10)
└── filter-result__body__right p → aantal waarderingen
```

We gebruiken **CSS selectors** (`.class-naam`) om elk stukje data op te halen.

In [ ]:
card = cards[0]

# Naam
naam = card.select_one(".filter-result__name").get_text(strip=True)
print(f"Naam: {naam}")

# Link naar profiel
link = card.select_one(".filter-result__name")["href"]
print(f"Link: https://www.zorgkaartnederland.nl{link}")

# Praktijk + locatie
plaats_el = card.select_one(".filter-result__places")
praktijk = plaats_el.get_text(strip=True)
print(f"Praktijk: {praktijk}")

# Latitude en longitude zitten als data-attribuut op het element!
lat_lon = plaats_el["data-location"]
print(f"Coördinaten: {lat_lon}")

# Score
score = card.select_one(".filter-result__score").get_text(strip=True)
print(f"Score: {score}")

# Aantal waarderingen
review_text = card.select_one(".filter-result__body__right p").get_text(strip=True)
n_reviews = review_text.split("waarderin")[0].strip()
print(f"Aantal waarderingen: {n_reviews}")

## Stap 4: Alle huisartsen van één pagina scrapen

We stoppen de logica van hierboven in een functie die een volledige pagina scraped en een lijst van dictionaries teruggeeft.

In [ ]:
def scrape_pagina(url):
    """Scrape alle huisartsen van één Zorgkaart-pagina."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")
    
    resultaten = []
    for card in soup.select(".filter-result"):
        naam_el = card.select_one(".filter-result__name")
        if not naam_el:
            continue
        
        plaats_el = card.select_one(".filter-result__places")
        score_el = card.select_one(".filter-result__score")
        review_el = card.select_one(".filter-result__body__right p")
        
        resultaten.append({
            "naam": naam_el.get_text(strip=True),
            "score": score_el.get_text(strip=True) if score_el else None,
            "aantal_waarderingen": review_el.get_text(strip=True).split("waarderin")[0].strip() if review_el else None,
            "praktijk": plaats_el.get_text(strip=True) if plaats_el else None,
            "lat_lon": plaats_el.get("data-location", "") if plaats_el else "",
            "profiel_url": "https://www.zorgkaartnederland.nl" + naam_el.get("href", ""),
        })
    
    return resultaten

# Test!
resultaten = scrape_pagina("https://www.zorgkaartnederland.nl/huisarts/amsterdam")
print(f"Gevonden: {len(resultaten)} huisartsen")
resultaten[:3]

## Stap 5: Meerdere pagina's scrapen

Eén pagina toont 20 huisartsen. Amsterdam heeft er 436, verdeeld over 22 pagina's. De URL voor pagina 2 is simpelweg:

```
/huisarts/amsterdam/pagina2
```

We bouwen een functie die meerdere pagina's achter elkaar scraped. Met een `time.sleep()` tussen de requests zijn we netjes voor de server.

In [ ]:
import time

def scrape_stad(stad, max_paginas=5):
    """Scrape meerdere pagina's huisartsen voor een stad."""
    alle_resultaten = []
    
    for pagina in range(1, max_paginas + 1):
        if pagina == 1:
            url = f"https://www.zorgkaartnederland.nl/huisarts/{stad}"
        else:
            url = f"https://www.zorgkaartnederland.nl/huisarts/{stad}/pagina{pagina}"
        
        resultaten = scrape_pagina(url)
        
        if not resultaten:  # geen resultaten meer → we zijn voorbij de laatste pagina
            break
        
        alle_resultaten.extend(resultaten)
        print(f"  Pagina {pagina}: {len(resultaten)} huisartsen")
        
        time.sleep(0.5)  # wees netjes: 0.5 seconde pauze tussen requests
    
    return alle_resultaten

print("Scraping Amsterdam (max 5 pagina's)...")
amsterdam = scrape_stad("amsterdam", max_paginas=5)
print(f"\nTotaal: {len(amsterdam)} huisartsen")

## Stap 6: Meerdere steden vergelijken

Nu wordt het leuk. We scrapen de 6 grootste steden en voegen een kolom `stad` toe zodat we straks kunnen vergelijken.

In [ ]:
import pandas as pd

steden = {
    "amsterdam": "Amsterdam",
    "rotterdam": "Rotterdam",
    "den-haag": "Den Haag",
    "utrecht": "Utrecht",
    "eindhoven": "Eindhoven",
    "leiden": "Leiden",
}

alle_data = []
for slug, naam in steden.items():
    print(f"\n📍 {naam}:")
    resultaten = scrape_stad(slug, max_paginas=5)
    for r in resultaten:
        r["stad"] = naam
    alle_data.extend(resultaten)

df = pd.DataFrame(alle_data)

# Datatypes goed zetten
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["aantal_waarderingen"] = pd.to_numeric(df["aantal_waarderingen"], errors="coerce")

# Lat/lon splitsen in twee kolommen
df[["lat", "lon"]] = df["lat_lon"].str.split(",", expand=True).astype(float)
df = df.drop(columns=["lat_lon"])

print(f"\n✅ Totaal: {len(df)} huisartsen in {df['stad'].nunique()} steden")
df.head(10)

## Stap 7: Data verkennen

Laten we eens kijken wat we hebben.

In [ ]:
df.info()
print()
df.describe()

In [ ]:
# Hoeveel huisartsen per stad?
df["stad"].value_counts()

## Stap 8: Visualiseren

### Scoreverdeling per stad

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

# Boxplot van scores per stad
df.boxplot(column="score", by="stad", ax=ax, grid=False)
ax.set_title("Verdeling huisartsscores per stad")
ax.set_xlabel("")
ax.set_ylabel("Score")
plt.suptitle("")  # verwijder automatische titel
plt.tight_layout()
plt.show()

### Gemiddelde score per stad

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

gem_scores = df.groupby("stad")["score"].mean().sort_values()
bars = ax.barh(gem_scores.index, gem_scores.values, color="#2196F3")

# Cijfer naast elke balk
for bar, score in zip(bars, gem_scores.values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{score:.2f}", va="center", fontweight="bold")

ax.set_xlim(gem_scores.min() - 0.3, 10.2)
ax.set_xlabel("Gemiddelde score")
ax.set_title("Gemiddelde huisartsscore per stad")
plt.tight_layout()
plt.show()

### Top 10 en Bottom 10 huisartsen

We filteren op minstens 3 waarderingen, anders is het cijfer niet zo betrouwbaar.

In [ ]:
betrouwbaar = df[df["aantal_waarderingen"] >= 3].copy()
print(f"{len(betrouwbaar)} huisartsen met ≥3 waarderingen (van {len(df)} totaal)\n")

print("🏆 TOP 10:")
print(betrouwbaar.nlargest(10, "score")[["naam", "stad", "score", "aantal_waarderingen", "praktijk"]].to_string(index=False))

print("\n📉 BOTTOM 10:")
print(betrouwbaar.nsmallest(10, "score")[["naam", "stad", "score", "aantal_waarderingen", "praktijk"]].to_string(index=False))

### Huisartsen op de kaart

We hebben lat/lon meegescrapet — laten we daar iets mee doen. Scores als kleur op een scatterplot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

scatter = ax.scatter(
    df["lon"], df["lat"],
    c=df["score"],
    cmap="RdYlGn",       # rood = laag, groen = hoog
    vmin=5, vmax=10,
    s=30, alpha=0.7, edgecolors="white", linewidth=0.3,
)

plt.colorbar(scatter, ax=ax, label="Score", shrink=0.6)

# Labels per stad (gemiddelde positie)
for stad, groep in df.groupby("stad"):
    ax.annotate(
        stad,
        (groep["lon"].mean(), groep["lat"].mean()),
        fontsize=12, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Huisartsen in Nederland — score per locatie")
plt.tight_layout()
plt.show()

### Score vs. aantal waarderingen

Krijgen populaire huisartsen lagere scores? Of juist hogere?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df["aantal_waarderingen"], df["score"], alpha=0.5, s=40, c="#2196F3", edgecolors="white", linewidth=0.3)
ax.set_xlabel("Aantal waarderingen")
ax.set_ylabel("Score")
ax.set_title("Score vs. aantal waarderingen — meer reviews = lager cijfer?")
plt.tight_layout()
plt.show()

# Correlatie
corr = df[["score", "aantal_waarderingen"]].corr().iloc[0, 1]
print(f"Correlatie: {corr:.3f}")

## Stap 9: Data opslaan

We slaan het resultaat op als CSV, zodat we het later kunnen hergebruiken zonder opnieuw te hoeven scrapen.

In [ ]:
df.to_csv("huisartsen_zorgkaart.csv", index=False)
print(f"Opgeslagen: huisartsen_zorgkaart.csv ({len(df)} rijen)")

## Recap

| Wat | Hoe |
|-----|-----|
| Webpagina ophalen | `requests.get(url)` |
| HTML parsen | `BeautifulSoup(html, "lxml")` |
| Elementen zoeken | `soup.select(".class-naam")` |
| Tekst uitlezen | `element.get_text(strip=True)` |
| Attributen uitlezen | `element["data-location"]` |
| Meerdere pagina's | Loop + `time.sleep()` |
| Data structureren | `pd.DataFrame(lijst_van_dicts)` |

### Zelf verder experimenteren?

Probeer eens:
- Andere zorgverleners te scrapen: `/tandarts`, `/fysiotherapeut`, `/psycholoog`
- Meer steden toe te voegen
- De detailpagina per huisarts te scrapen voor meer info
- Sorteren op meeste reviews: voeg `?sort=waarderingen-desc` toe aan de URL